# Lab 1: Non-instantaneous populations and X-ray luminosity functions

#### Anastasios Fragos — POSYDON School 2026, Wednesday 26 August, 10:00–11:00

This lab turns a POSYDON population into an observable prediction: the X-ray
luminosity function (XLF) of a star-forming galaxy. It is written both as a
one-hour guided exercise and as self-contained material for later offline use.

**Audience.** Participants should be comfortable with Python, NumPy, pandas,
and the basic POSYDON population workflow from Monday. No prior X-ray
astronomy is required.

**By the end you will be able to:**

- explain why XRBs are non-instantaneous sources and distinguish snapshot from
  full-history population sampling;
- run a small two-metallicity population with POSYDON 2.3;
- identify BH/NS XRB candidates and consistently label donor and accretor;
- calculate classical RLO, wind-fed, and empirical Be-XRB luminosities;
- construct and normalize a cumulative XLF per unit star-formation rate; and
- interpret the roles of compact-object type, accretion mode, and metallicity.

**Classroom route (60 min).** Complete Sections 1–7 through Exercise 5. The
low-metallicity and observational comparisons in Section 8 are optional
extensions. Collapsed solution cells establish the canonical downstream state,
so the notebook also works with **Restart Kernel and Run All Cells**.


## 1. Why X-ray binaries require a different population view

A supernova or compact-object merger is treated as an event: its duration is
negligible compared with stellar-evolution timescales. An X-ray binary is a
*phase*. Its mass-transfer rate, orbit, donor, and luminosity evolve while it
remains observable.

There are two common sampling strategies:

1. **Snapshot:** assign birth times, evolve every binary to a chosen galaxy
   age, and count the XRBs present then. This is intuitive and is our method
   today.
2. **Full history:** retain every XRB interval and weight each state by its
   duration. This uses the simulation more efficiently but requires explicit
   time integration; it is deferred to a future XRB tutorial.

![Timeline comparing snapshot and full-history methods](xrb_schematic.png)

The snapshot answers: *What would an observer see if the galaxy were observed
now?* It does not reconstruct every X-ray episode a binary experienced.


### 1.1 The three accretion channels used here

- **Roche-lobe overflow (RLO):** the donor fills its Roche lobe and matter
  flows through the inner Lagrange point. The POSYDON mass-transfer rate is
  used as the supplied accretion rate.
- **Wind-fed XRB:** a detached compact object captures part of a stellar wind.
  We estimate the wind velocity and use an orbit-averaged
  Bondi–Hoyle–Lyttleton (BHL) capture rate.
- **Be-XRB:** a rapidly rotating B star supplies a decretion disc. After the
  notebook identifies plausible Be systems, POSYDON's empirical
  period–luminosity relation supplies their X-ray luminosity.

These labels describe the assumed fuel supply, not three different compact
objects. Both neutron stars (NSs) and black holes (BHs) may appear in the
subpopulations.


### 1.2 From accretion rate to luminosity

For radiative efficiency \(\eta\), a sub-Eddington flow has

\[
L_{\rm bol}=\eta\,\dot M c^2.
\]

The Eddington luminosity follows from balance between gravity and radiation
pressure on ionized gas,

\[
L_{\rm Edd}=\frac{4\pi GMc}{0.2(1+X_{\rm surf})}, \qquad
\dot M_{\rm Edd}=\frac{L_{\rm Edd}}{\eta c^2}.
\]

For \(\dot m=\dot M/\dot M_{\rm Edd}>1\), we use the classical
Shakura–Sunyaev/King prescription. The apparent isotropic luminosity grows
logarithmically and is enhanced by geometrical beaming. The beaming factor is
one through \(\dot m=8.5\), then \(b=73/\dot m^2\), with a floor
\(b_{\min}=3.2\times10^{-3}\).

The POSYDON utility returns a **bolometric, isotropic-equivalent** luminosity.
For the RLO and wind-fed curves we explicitly adopt
\(L_{0.5-8\,\mathrm{keV}}=0.5L_{\rm bol}\). This approximate band correction is
a notebook-level observational assumption, not part of the reusable physics.


<div class="alert alert-info">

### Exercise 1 — Predict before calculating

At the same \(\dot M\), which system should generally have the larger
radiative efficiency: a compact NS with a radius near 12.5 km, or a
non-spinning BH? What changes once the source is strongly super-Eddington?

Write down your prediction. We will revisit it after splitting the XLF.

</div>


## 2. Reproducible POSYDON 2.3 setup

The cells below deliberately check capabilities rather than comparing a
version string only: development builds from the `v2.3` branch may have a
`.dev` version. A successful import of the new public API is the decisive
compatibility check.


In [ ]:
from pathlib import Path
import contextlib
import json
import re
import shutil
import subprocess

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import posydon
from posydon.config import PATH_TO_POSYDON, PATH_TO_POSYDON_DATA
from posydon.popsyn.synthetic_population import Population, PopulationRunner
from posydon.utils.xrb import (
    accretion_luminosity,
    be_xray_luminosity,
    black_hole_radiative_efficiency,
    bondi_hoyle_accretion_rate,
    neutron_star_radiative_efficiency,
    wind_velocity,
)
from posydon.utils import constants as const
from posydon.utils.common_functions import (
    orbital_separation_from_period,
    roche_lobe_radius,
)

mpl.rcParams["text.usetex"] = False
mpl.rcParams["font.family"] = "DejaVu Serif"
pd.set_option("display.max_columns", 18)

print("POSYDON version:", posydon.__version__)
print("POSYDON code:", PATH_TO_POSYDON)
print("POSYDON data:", PATH_TO_POSYDON_DATA)

try:
    POSYDON_SHA = subprocess.run(
        ["git", "-C", PATH_TO_POSYDON, "rev-parse", "HEAD"],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
except (OSError, subprocess.CalledProcessError):
    POSYDON_SHA = None

if posydon.__version__ != "unknown" and not str(
        posydon.__version__).startswith("2.3"):
    raise RuntimeError(
        "This tutorial requires POSYDON 2.3; found "
        f"{posydon.__version__!r}."
    )
print("POSYDON commit:", POSYDON_SHA or "not available in this installation")


The 2026 school archive is installed separately from the large DR2 grids:

```bash
get-posydon-data 2026_school_data
```

After installation, the archive contains the precomputed 100,000-binary
populations used in Section 4. The required small run in Section 3 uses the
installed DR2 grids directly.


In [ ]:
WORK_DIR = Path("xlf_lab_work")
WORK_DIR.mkdir(exist_ok=True)

SCHOOL_DATA = Path(PATH_TO_POSYDON_DATA).parent / "2026_school_data"
PRECOMPUTED_DIR = SCHOOL_DATA / "populations" / "XRB_100K_pops"

if not PRECOMPUTED_DIR.is_dir():
    raise FileNotFoundError(
        f"Missing {PRECOMPUTED_DIR}. Install it with "
        "`get-posydon-data 2026_school_data`."
    )

metadata = json.loads(
    (SCHOOL_DATA / "generation_metadata.json").read_text()
)
archive_is_v23 = (
    str(metadata["posydon_version"]).startswith("2.3")
    or metadata.get("posydon_branch") == "v2.3"
)
if not archive_is_v23 or metadata.get("posydon_dataset") != "DR2":
    raise RuntimeError("The school archive is not a POSYDON 2.3/DR2 dataset.")
if POSYDON_SHA and metadata.get("posydon_commit") != POSYDON_SHA:
    print("Note: this POSYDON commit differs from the archived generation SHA.")

print("Precomputed populations:", PRECOMPUTED_DIR)
print("Archive generation SHA:", metadata["posydon_commit"])


## 3. Required small population run

We evolve 100 binaries at each of \(Z/Z_\odot=1\) and 0.1. The constant
star-formation history distributes their birth times over 100 Myr, making the
final table a snapshot of a continuously star-forming galaxy at 100 Myr.

Rather than asking everyone to edit an `.ini` file manually, the next cell
copies the version-controlled POSYDON default and changes only four named
parameters. This preserves a complete record of all other assumptions.


<div class="alert alert-success">

### Exercise 2 — Configure the snapshot

Before revealing the solution, identify the four parameters controlling:

1. population size,
2. metallicities,
3. star-formation history, and
4. snapshot age.

</div>


In [ ]:
# Write the four parameter names here before revealing the solution.


In [ ]:
default_ini = Path(PATH_TO_POSYDON) / "posydon/popsyn/population_params_default.ini"
live_ini = WORK_DIR / "population_params_xrb_100.ini"
text = default_ini.read_text()

updates = {
    "number_of_binaries": "100",
    "metallicities": "[1.0, 0.1]",
    "star_formation": "'constant'",
    "max_simulation_time": "1.0e8",
}
for key, value in updates.items():
    pattern = rf"(?m)^(\s*{key}\s*=\s*).*$"
    text, count = re.subn(pattern, rf"\g<1>{value}", text, count=1)
    if count != 1:
        raise RuntimeError(f"Could not update {key!r} in {default_ini}")

live_ini.write_text(text)
print(live_ini.read_text().split("[BinaryPopulation_options]", 1)[1][:700])


Running 200 binaries is intentionally part of the classroom exercise. It
demonstrates the real workflow, but it is far too small for a smooth XLF. The
100,000-binary archive will replace it for the scientific plots.


In [ ]:
RUN_LIVE_POPULATION = True

if RUN_LIVE_POPULATION:
    live_ini_path = live_ini.resolve()
    with contextlib.chdir(WORK_DIR):
        poprun = PopulationRunner(str(live_ini_path), verbose=True)
        poprun.evolve(overwrite=True)
else:
    print("Live run skipped deliberately.")


Inspect one of the newly generated files. Counts and exact pathways fluctuate
with the random seed, which is a useful reminder that a 100-system population
is a demonstration, not a converged prediction.


In [ ]:
live_solar_path = Path("1e+00_Zsun_population.h5")
if not live_solar_path.exists():
    candidates = list(WORK_DIR.glob("**/1e+00_Zsun_population.h5"))
    if candidates:
        live_solar_path = candidates[0]

live_pop = Population(str(live_solar_path))
print("Systems:", live_pop.number_of_systems)
print(live_pop.mass_per_metallicity)
display(live_pop.oneline.head(3))


## 4. Load a statistically useful snapshot

We now switch to fresh POSYDON 2.3 populations containing 100,000 binaries at
solar and 10% solar metallicity. The archive contains the `.ini`, POSYDON
commit, DR2 dataset identifier, generation log, and checksums used to create
them.


In [ ]:
population_paths = {
    1.0: PRECOMPUTED_DIR / "1e+00_Zsun_population.h5",
    0.1: PRECOMPUTED_DIR / "1e-01_Zsun_population.h5",
}
for metallicity, path in population_paths.items():
    if not path.is_file():
        raise FileNotFoundError(f"Missing Z/Zsun={metallicity}: {path}")

solar_pop = Population(str(population_paths[1.0]))
print("Solar systems:", solar_pop.number_of_systems)
print(solar_pop.mass_per_metallicity)


<div class="alert alert-success">

### Exercise 3 — Select candidate XRBs

Build a mask that requires exactly one BH or NS and one non-degenerate donor.
Also reject disrupted, failed (`ERR`), and initially overflowing systems.

Why is “contains a compact object” insufficient? Because DCOs, WDs, and
disrupted pairs do not represent the accretor-plus-normal-donor systems modeled
by the luminosity functions below.

</div>


In [ ]:
# Build `selected_indices` from `solar_pop.oneline`.


In [ ]:
state_columns = ["state_f", "S1_state_f", "S2_state_f"]
final_states = solar_pop.oneline.select(columns=state_columns)

compact = {"BH", "NS"}
degenerate_or_absent = {"BH", "NS", "WD", "massless_remnant"}
bad_binary_states = {"disrupted", "ERR", "initial_RLOF"}

s1_compact = final_states["S1_state_f"].isin(compact)
s2_compact = final_states["S2_state_f"].isin(compact)
s1_is_donor = ~final_states["S1_state_f"].isin(degenerate_or_absent)
s2_is_donor = ~final_states["S2_state_f"].isin(degenerate_or_absent)
bound_and_valid = ~final_states["state_f"].isin(bad_binary_states)

candidate_mask = ((s1_compact & s2_is_donor)
                  | (s2_compact & s1_is_donor)) & bound_and_valid
selected_indices = final_states.index[candidate_mask].to_list()

print(f"Selected {len(selected_indices):,} of {len(final_states):,} systems")


Exporting the smaller selection makes repeated analysis faster and leaves the
downloaded school archive untouched. `overwrite=True` also makes this cell
idempotent during notebook development.


In [ ]:
selected_solar_path = WORK_DIR / "XRB_candidates_1e+00_Zsun.h5"
solar_pop.export_selection(
    selected_indices, str(selected_solar_path), append=False, overwrite=True
)
xrb_candidates = Population(str(selected_solar_path))
print("Candidate systems:", xrb_candidates.number_of_systems)


## 5. Map S1/S2 to donor/accretor and calculate luminosities

POSYDON tracks stars as S1 and S2, but accretion physics is clearer in terms
of donor and accretor. Our selection guarantees exactly one BH/NS, so the
mapping is unambiguous.

The function below remains in the notebook by design. It makes pedagogical
choices—what counts as an XRB, how a Be star is identified, which snapshot
columns are retained, and how a 0.5–8 keV correction is applied. Those choices
are not universal POSYDON physics.


<div class="alert alert-warning">

### Be-XRB identification used in this lab

A detached candidate is labeled Be when the donor is an H-rich core-H-burning
star with \(M\ge6\,M_\odot\), surface rotation at least 70% of critical,
\(10\le P_{\rm orb}\le300\) days, and a notional decretion disc extending to
100 stellar radii beyond the donor's periastron Roche lobe. We assign a 10%
duty cycle when constructing the XLF.

This is a model choice and a major source of uncertainty—not a definition of
all observed Be-XRBs.

</div>


In [ ]:
def xrb_snapshot_selection(history_chunk, oneline_chunk,
                           formation_channels_chunk=None):
    # Return selected snapshot XRB properties and classical luminosities.
    df = oneline_chunk.copy()
    compact = {"BH", "NS"}
    excluded = {"BH", "NS", "WD", "massless_remnant"}
    bad_states = {"disrupted", "ERR", "initial_RLOF"}

    s1_compact = df["S1_state_f"].isin(compact)
    s2_compact = df["S2_state_f"].isin(compact)
    keep = (((s1_compact & ~df["S2_state_f"].isin(excluded))
             | (s2_compact & ~df["S1_state_f"].isin(excluded)))
            & ~df["state_f"].isin(bad_states))
    df = df.loc[keep].copy()
    s1_compact = df["S1_state_f"].isin(compact).to_numpy()

    def choose(s1_column, s2_column, accretor=True):
        use_s1 = s1_compact if accretor else ~s1_compact
        return np.where(use_s1, df[s1_column], df[s2_column])

    out = pd.DataFrame(index=df.index)
    out["time"] = df["time_f"].to_numpy() * 1e-6  # Myr
    out["metallicity"] = df["metallicity"].to_numpy()
    out["binary_state"] = df["state_f"].to_numpy()
    out["orbital_period"] = df["orbital_period_f"].to_numpy(float)
    out["eccentricity"] = df["eccentricity_f"].to_numpy(float)

    out["accretor_state"] = choose("S1_state_f", "S2_state_f")
    out["accretor_mass"] = choose("S1_mass_f", "S2_mass_f").astype(float)
    out["accretor_radius"] = 10.0**choose(
        "S1_log_R_f", "S2_log_R_f").astype(float)
    out["accretor_spin"] = choose("S1_spin_f", "S2_spin_f").astype(float)
    out["donor_state"] = choose(
        "S1_state_f", "S2_state_f", accretor=False)
    out["donor_mass"] = choose(
        "S1_mass_f", "S2_mass_f", accretor=False).astype(float)
    out["donor_radius"] = 10.0**choose(
        "S1_log_R_f", "S2_log_R_f", accretor=False).astype(float)
    out["donor_luminosity"] = 10.0**choose(
        "S1_log_L_f", "S2_log_L_f", accretor=False).astype(float)
    out["donor_surface_h1"] = choose(
        "S1_surface_h1_f", "S2_surface_h1_f", accretor=False).astype(float)
    donor_he_core_mass = choose(
        "S1_he_core_mass_f", "S2_he_core_mass_f", accretor=False).astype(float)
    # A missing core mass for an unevolved donor means that no resolved helium
    # core is present yet; use zero rather than discarding the whole row.
    out["donor_he_core_mass"] = np.where(
        np.isfinite(donor_he_core_mass) & (donor_he_core_mass >= 0.0),
        donor_he_core_mass, 0.0,
    )
    out["donor_rotation_fraction"] = choose(
        "S1_surf_avg_omega_div_omega_crit_f",
        "S2_surf_avg_omega_div_omega_crit_f",
        accretor=False,
    ).astype(float)
    donor_log_wind = choose(
        "S1_lg_wind_mdot_f", "S2_lg_wind_mdot_f", accretor=False
    ).astype(float)
    out["donor_wind_mass_loss"] = 10.0**donor_log_wind
    out["rlo_mass_transfer_rate"] = 10.0**df[
        "lg_mtransfer_rate_f"].to_numpy(float)

    separation = orbital_separation_from_period(
        out["orbital_period"], out["accretor_mass"], out["donor_mass"]
    )
    out["orbital_separation"] = separation
    donor_roche_lobe_periastron = roche_lobe_radius(
        out["donor_mass"], out["accretor_mass"],
        separation * (1.0 - out["eccentricity"]),
    )

    detached = out["binary_state"].eq("detached")
    rlo = out["binary_state"].isin(["RLO1", "RLO2"])
    be = (detached
          & out["donor_state"].eq("H-rich_Core_H_burning")
          & (out["donor_mass"] >= 6.0)
          & (out["donor_rotation_fraction"] >= 0.7)
          & out["orbital_period"].between(10.0, 300.0)
          & (donor_roche_lobe_periastron <= 100.0 * out["donor_radius"]))
    wind = detached & ~be
    out["accretion_mode"] = np.select(
        [be, rlo, wind], ["Be", "RLO", "wind"], default="other"
    )

    bh = out["accretor_state"].eq("BH").to_numpy()
    # Missing BH spins are explicitly treated as zero-spin. Missing NS radii
    # are replaced by the POSYDON default 12.5 km.
    bh_spin = out["accretor_spin"].to_numpy(float)
    bh_spin = np.where(np.isfinite(bh_spin), bh_spin, 0.0)
    ns_radius = out["accretor_radius"].to_numpy(float)
    ns_radius = np.where(np.isfinite(ns_radius) & (ns_radius > 0.0),
                         ns_radius, 1.25e6 / const.Rsun)
    efficiency = np.where(
        bh,
        black_hole_radiative_efficiency(bh_spin),
        neutron_star_radiative_efficiency(out["accretor_mass"], ns_radius),
    )
    out["radiative_efficiency"] = efficiency

    speed = wind_velocity(
        out["donor_mass"], out["donor_radius"], out["donor_luminosity"],
        out["donor_wind_mass_loss"], out["donor_surface_h1"],
        out["donor_he_core_mass"], scheme="Kudritzki+2000",
    )
    wind_rate = bondi_hoyle_accretion_rate(
        out["accretor_mass"], out["donor_mass"],
        out["donor_wind_mass_loss"], out["orbital_separation"],
        out["eccentricity"], speed,
    )
    supplied_rate = np.where(rlo, out["rlo_mass_transfer_rate"], wind_rate)
    luminosity, beaming, eddington_ratio = accretion_luminosity(
        supplied_rate, out["accretor_mass"], out["donor_surface_h1"],
        efficiency,
    )

    out["L_bol_iso"] = luminosity
    out["Lx_0p5_8keV"] = 0.5 * luminosity
    out["beaming_factor"] = beaming
    out["eddington_ratio"] = eddington_ratio
    out.loc[be, "Lx_0p5_8keV"] = be_xray_luminosity(
        out.loc[be, "orbital_period"]
    )
    out.loc[be, "beaming_factor"] = 1.0
    out["duty_cycle"] = np.where(be, 0.1, 1.0)

    if formation_channels_chunk is not None:
        out["formation_channel"] = formation_channels_chunk.loc[
            out.index, "channel"]
    return out.loc[out["accretion_mode"] != "other"]


Test one chunk before applying the function to the whole selection. Read the
columns horizontally: this is the bridge between binary evolution and the
observable XLF.


In [ ]:
example = xrb_snapshot_selection(
    xrb_candidates.history[0], xrb_candidates.oneline[0], None
)
display(example.T)


<div class="alert alert-success">

### Exercise 4 — Sanity-check the physics

Inspect the example above.

1. Is the compact object labeled as the accretor regardless of whether it is
   S1 or S2?
2. Does the accretion mode agree with the final binary state?
3. If `eddington_ratio > 8.5`, is `beaming_factor < 1`?
4. Which assumptions—not simulation outputs—entered `Lx_0p5_8keV`?

</div>


In [ ]:
xrb_population = xrb_candidates.create_transient_population(
    xrb_snapshot_selection,
    "XRB_snapshot",
)
if xrb_population is None:
    raise RuntimeError("The selected population contains no luminous XRB states.")

display(xrb_population.population.head())
print(xrb_population.population["accretion_mode"].value_counts())


## 6. Construct the cumulative X-ray luminosity function

The differential XLF \(dN/dL_X\) counts systems per luminosity interval. We
will use the cumulative form

\[
N(>L_X)=\int_{L_X}^{\infty}\frac{dN}{dL}\,dL,
\]

which avoids arbitrary luminosity bins and is commonly plotted in log–log
space. Raw counts are not physical: simulating ten times more binaries would
produce roughly ten times more XRBs. Each POSYDON system therefore receives a
probability per unit stellar mass formed.

For a constant \(1\,M_\odot\,\mathrm{yr}^{-1}\) star-formation rate over
100 Myr, the underlying formed mass is \(10^8\,M_\odot\). We also multiply by
the King orientation probability and the Be duty cycle.


In [ ]:
population_parameters = xrb_population.ini_params.copy()
population_parameters.update({
    "primary_mass_min": 0.1,
    "q_min": 0.0,
    "q_max": 1.0,
})

model_weights = xrb_population.calculate_model_weights(
    model_weights_identifier="base_IMF",
    population_parameters=population_parameters,
)
if isinstance(model_weights, pd.DataFrame):
    model_weights = model_weights.iloc[:, 0]
model_weights = np.asarray(model_weights, dtype=float)

SFR = 1.0  # Msun / yr
TIME_WINDOW = 1.0e8  # yr
observability = (
    xrb_population.population["beaming_factor"].to_numpy(float)
    * xrb_population.population["duty_cycle"].to_numpy(float)
)
physical_weights = model_weights * SFR * TIME_WINDOW * observability


<div class="alert alert-success">

### Exercise 5 — Write a weighted cumulative distribution

Complete a helper that sorts luminosities and cumulatively sums weights from
the bright end. The returned \(y_i\) must represent the expected number of
sources with \(L_X\ge x_i\), not the number fainter than \(x_i\).

</div>


In [ ]:
# Define `weighted_xlf(luminosity, weights, minimum=1e35)`.


In [ ]:
def weighted_xlf(luminosity, weights, minimum=1.0e35):
    luminosity = np.asarray(luminosity, dtype=float)
    weights = np.asarray(weights, dtype=float)
    valid = (np.isfinite(luminosity) & np.isfinite(weights)
             & (luminosity >= minimum) & (weights >= 0.0))
    luminosity = luminosity[valid]
    weights = weights[valid]
    order = np.argsort(luminosity)
    x = luminosity[order]
    y = np.cumsum(weights[order][::-1])[::-1]
    return x, y


In [ ]:
luminosity = xrb_population.population["Lx_0p5_8keV"].to_numpy(float)
x, y = weighted_xlf(luminosity, physical_weights)

fig, ax = plt.subplots(figsize=(6.2, 4.8))
ax.step(x, y, where="post", color="black", label="All XRBs")
ax.set(xscale="log", yscale="log", xlim=(1e35, 1e41),
       xlabel=r"$L_{X,0.5-8\,\mathrm{keV}}\;[\mathrm{erg\,s^{-1}}]$",
       ylabel=r"$N(>L_X)/(M_\odot\,\mathrm{yr}^{-1})$")
ax.grid(alpha=0.2, which="both")
ax.legend()
plt.show()


### 6.1 Split the physical channels

The total XLF can hide compensating effects. We now split by compact-object
type and fuel supply. A curve can be absent simply because a 100,000-binary
Monte Carlo sample contains no systems above the luminosity threshold.


In [ ]:
fig, ax = plt.subplots(figsize=(7.0, 5.2))
styles = {
    ("BH", "RLO"): ("tab:blue", "-"),
    ("NS", "RLO"): ("tab:orange", "-"),
    ("BH", "wind"): ("tab:blue", "--"),
    ("NS", "wind"): ("tab:orange", "--"),
    ("BH", "Be"): ("tab:green", ":"),
    ("NS", "Be"): ("tab:red", ":"),
}

table = xrb_population.population
for (compact_type, mode), (color, linestyle) in styles.items():
    mask = (table["accretor_state"].eq(compact_type)
            & table["accretion_mode"].eq(mode)).to_numpy()
    x_sub, y_sub = weighted_xlf(luminosity[mask], physical_weights[mask])
    if len(x_sub):
        ax.step(x_sub, y_sub, where="post", color=color,
                linestyle=linestyle, label=f"{compact_type} + {mode}")

ax.set(xscale="log", yscale="log", xlim=(1e35, 1e41),
       xlabel=r"$L_{X,0.5-8\,\mathrm{keV}}\;[\mathrm{erg\,s^{-1}}]$",
       ylabel=r"$N(>L_X)/(M_\odot\,\mathrm{yr}^{-1})$")
ax.grid(alpha=0.2, which="both")
ax.legend(ncol=2, fontsize=9)
plt.show()


## 7. Interpret, do not merely plot

Return to your prediction from Exercise 1.

- NSs can have higher radiative efficiency than non-spinning BHs, but their
  smaller Eddington luminosities change where super-Eddington effects begin.
- RLO can sustain much larger supplied rates than wind capture, so it often
  controls the bright end.
- King beaming raises an on-axis isotropic-equivalent luminosity while reducing
  the probability that a randomly oriented observer lies inside the beam.
- Be-XRB numbers depend strongly on both the empirical selection and the
  assumed duty cycle.

The curves test the *combined model*: binary evolution, snapshot sampling,
accretion prescriptions, band correction, orientation, duty cycles, and IMF
normalization. Agreement or disagreement cannot be assigned to one ingredient
without controlled comparisons.


<div class="alert alert-info">

### Discussion checkpoint

Choose one bright system and trace every factor entering its final XLF weight
and luminosity. Which factors came from POSYDON evolution, which came from the
new luminosity module, and which were teaching-level observational choices?

</div>


## 8. Optional extension: metallicity and observations

Repeat Sections 4–6 with the \(0.1Z_\odot\) population. Lower-metallicity
massive stars generally lose less mass through winds and can form more massive
BHs, but the XLF response also depends on binary interaction and donor
structure. Treat this as a hypothesis to test, not a guaranteed monotonic
trend.

For an observational comparison, digitize or load a published XLF with its
bandpass and normalization recorded. Do **not** compare curves until you have
checked:

- energy band and absorption convention;
- differential versus cumulative definition;
- SFR calibration and IMF;
- luminosity completeness; and
- whether unresolved or background sources were removed.

A useful starting point is Figure 3 of Lehmer et al. (2021). The comparison is
left optional because a scientifically honest overlay requires the associated
observational table and uncertainty treatment, not pixels copied from a plot.


In [ ]:
# Optional scaffold: repeat the workflow for 0.1 Zsun.
low_z_pop = Population(str(population_paths[0.1]))
print(low_z_pop.number_of_systems)
print(low_z_pop.mass_per_metallicity)


## 9. Assumptions, pitfalls, and takeaways

### Assumptions made here

- A 100 Myr snapshot of a constant-SFR population.
- Classical RLO and orbit-averaged wind-fed accretion.
- Novikov–Thorne BH and Newtonian NS efficiencies.
- King super-Eddington beaming with a fixed minimum beaming factor.
- A factor 0.5 bolometric-to-0.5–8 keV conversion for RLO/wind systems.
- An empirical Be period–luminosity relation and 10% duty cycle.
- No GRRMHD, magnetic NS, wind-disc-formation, absorption, or detector model.

### Common pitfalls

- `lg_*` POSYDON columns are base-10 logarithms; convert them before passing
  rates or radii to the public physics API.
- Do not add donor wind loss to an RLO rate: the accretion channels are selected
  explicitly here.
- Sort luminosities and weights with the same index order.
- An isotropic-equivalent luminosity and a beaming probability are different
  quantities; apply both consistently.
- A smooth-looking XLF can still be statistically unconverged. Compare several
  population sizes or random seeds in research work.

### Takeaways

1. XRBs are evolving phases, so the sampling method is part of the prediction.
2. POSYDON evolves the binaries; a transparent post-processing model connects
   accretion properties to X-ray luminosity.
3. Physical population weights, orientation, and duty cycle turn simulated
   systems into an expected galactic XLF.
4. Splitting the XLF by compact object and accretion channel is essential for
   diagnosing the underlying binary physics.

### Further reading

- Misra et al. (2023), *A&A* 672, A99 — detailed HMXB XLF modeling.
- King (2008), *MNRAS* 385, L113 — super-Eddington beaming.
- Hurley, Tout & Pols (2002), *MNRAS* 329, 897 — binary-evolution wind capture.
- Dai, Liu & Li (2006), *ApJ* 653, 1410 — Be-XRB period relation.
- Lehmer et al. (2021), *ApJ* 907, 17 — observed metallicity-dependent XLFs.

The other Wednesday labs build on related population-synthesis ideas,
including reweighting across metallicity and connecting transient
populations to cosmological rates and detectability.
